In [ ]:
!pip install openpyxl -q

In [28]:
import os
import re
import numpy as np
import pandas as pd
from typing import List, Dict, Callable
from google.colab import files

In [29]:
def normalize_digits_fa2en(text: str) -> str:
    if text is None: return ""
    if isinstance(text, float) and np.isnan(text): return ""
    s = str(text)
    trans = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
    return s.translate(trans)

def normalize_persian_text(text: str) -> str:
    if text is None: return ""
    if isinstance(text, float) and np.isnan(text): return ""
    s = str(text)
    s = s.replace("\u200c"," ").replace("‌"," ")
    s = s.replace("ي","ی").replace("ك","ک")
    s = normalize_digits_fa2en(s)
    s = s.lower()
    s = re.sub(r"\s+"," ", s).strip()
    return s

def contains_any(text: str, patterns: List[str]) -> bool:
    return any(p in text for p in patterns)

In [ ]:
def get_col(df: pd.DataFrame, possible: List[str], default_value: str = "") -> pd.Series:
    for name in possible:
        if name in df.columns:
            return df[name]
    return pd.Series([default_value]*len(df), index=df.index)

def load_excel_or_csv(path: str) -> pd.DataFrame:
    
    try:
        df = pd.read_excel(path, engine="openpyxl")
        print(f"[INFO] Loaded Excel: {path} shape={df.shape}")
        return df
    except Exception as e_xlsx:
        print(f"[WARN] Not Excel or failed: {path} -> {e_xlsx}")
        for enc in ["utf-8-sig","utf-8","cp1256","windows-1256"]:
            try:
                df = pd.read_csv(path, encoding=enc, engine="python", on_bad_lines="skip")
                print(f"[INFO] Loaded CSV: {path} enc={enc} shape={df.shape}")
                return df
            except Exception as e_csv:
                print(f"[WARN] CSV read failed enc={enc}: {e_csv}")
        print(f"[ERROR] Could not read file: {path}")
        return pd.DataFrame()

def unify_generic(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
 
    if df.empty: return df
    out = pd.DataFrame(index=df.index)
    out["source"] = source_name

    out["title"]          = get_col(df, ["title","عنوان"]).astype(str)
    out["body_status"]    = get_col(df, ["body_status","وضعیت بدنه"], "").fillna("").astype(str)
    out["engine_status"]  = get_col(df, ["engine_status","وضعیت فنی"], "").fillna("").astype(str)
    out["chassis_status"] = get_col(df, ["chassis_status","وضعیت شاسی"], "").fillna("").astype(str)
    out["description"]    = get_col(df, ["description","توضیحات","توضیحات آگهی"], "").fillna("").astype(str)

    out["year_raw"]  = get_col(df, ["year","سال تولید"]).astype(str)

    out["price_raw"] = get_col(df, ["price_raw","قیمت","price"]).astype(str)

    out["price"] = pd.to_numeric(get_col(df, ["price","قیمت"], default_value=np.nan), errors="coerce")
    out["mileage_km"] = pd.to_numeric(get_col(df, ["mileage_km","کارکرد"], default_value=np.nan), errors="coerce")
    return out


In [31]:
def has_finance_or_dealer(text: str) -> bool:
    t = normalize_persian_text(text)
    bad = ["قسط","پیش قسط","پیش‌قسط","پیش پرداخت","پیش‌پرداخت","اقساط","نمایندگی","نمایشگاه","تماس با آقای","تماس با خانم"]
    return any(b in t for b in bad)

def filter_finance_and_dealer(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    combo = (df["title"].fillna("") + " " + df["description"].fillna("")).astype(str)
    mask = combo.apply(has_finance_or_dealer)
    print(f"[INFO] Removing {int(mask.sum())} finance/dealer-like ads")
    return df.loc[~mask].copy()


In [32]:
def is_negotiable_row(row: pd.Series) -> bool:
    texts = []
    for col in ["price_raw","price","title","description"]:
        if col in row.index:
            texts.append(str(row[col]))
    t = normalize_persian_text(" ".join(texts))

    return ("توافق" in t) or ("negotiable" in t)

def filter_negotiable(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    mask = df.apply(is_negotiable_row, axis=1)
    print(f"[INFO] Removing {int(mask.sum())} negotiable-price ads")
    return df.loc[~mask].copy()

In [33]:
def build_text_all(row: pd.Series) -> str:
    parts = [row.get("title",""), row.get("body_status",""), row.get("engine_status",""),
             row.get("chassis_status",""), row.get("description","")]
    return normalize_persian_text(" ".join([str(p) for p in parts if isinstance(p,str)]))

def add_text_all(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    df = df.copy()
    df["text_all"] = df.apply(build_text_all, axis=1)
    return df


In [34]:
def feat_darb_rang(r): t=r["text_all"]; p=["یک درب رنگ","1 درب رنگ","۱ درب رنگ","دو درب رنگ","2 درب رنگ","۲ درب رنگ","در جلو رنگ","در عقب رنگ","درب رنگ","در رنگ"]; return int(contains_any(t,p))
def feat_darb_taviz(r): t=r["text_all"]; p=["درب تعویض","در تعویض","درها تعویض","درها عوض شده","درب عوض شده"]; return int(contains_any(t,p))
def feat_darb_bedun_rang(r): t=r["text_all"]; p=["درب بدون رنگ","در بدون رنگ","درب ها بدون رنگ","درها بدون رنگ","درب بی رنگ","در بی رنگ"]; return int(contains_any(t,p))
def feat_kaput_taviz(r): t=r["text_all"]; p=["کاپوت تعویض","تعویض کاپوت","کاپوت عوض شده"]; return int(contains_any(t,p))
def feat_kaput_bedun_rang(r): t=r["text_all"]; p=["کاپوت بدون رنگ","کاپوت بی رنگ","بدون رنگ کاپوت"]; return int(contains_any(t,p))
def feat_darb_sandogh_rang(r): t=r["text_all"]; p=["درب صندوق رنگ","در صندوق رنگ","صندوق رنگ"]; return int(contains_any(t,p) and "صندوق بدون رنگ" not in t and "صندوق بی رنگ" not in t)
def feat_darb_sandogh_bedun_rang(r): t=r["text_all"]; p=["درب صندوق بدون رنگ","در صندوق بدون رنگ","صندوق بدون رنگ","صندوق بی رنگ"]; return int(contains_any(t,p))
def feat_shasi_zarbe(r): t=r["text_all"]; p=["شاسی ضربه","ضربه به شاسی","شاسی خورده","شاسی کج","شاسی آسیب","شاسی شکسته"]; return int(contains_any(t,p))
def feat_shasi_pelomp(r): t=r["text_all"]; p=["شاسی پلمپ","پلمپ شاسی","سالم و پلمپ"]; return int(contains_any(t,p))
def feat_shasi_salem(r): t=r["text_all"]; p=["شاسی سالم","شاسی ها سالم","شاسی بی ضربه","شاسی ها بی ضربه","شاسی دست نخورده"]; return int(contains_any(t,p))
def feat_saghf_rang(r): t=r["text_all"]; p=["سقف رنگ","رنگ سقف"]; return int(contains_any(t,p) and "سقف بدون رنگ" not in t and "سقف بی رنگ" not in t)
def feat_saghf_bedun_rang(r): t=r["text_all"]; p=["سقف بدون رنگ","سقف بی رنگ"]; return int(contains_any(t,p))
def feat_sotun_salem(r): t=r["text_all"]; p=["ستون سالم","ستون ها سالم","ستون بی ضربه","ستون ها بی ضربه","ستون دست نخورده"]; return int(contains_any(t,p))
def feat_sotun_zarbe(r): t=r["text_all"]; p=["ستون ضربه","ضربه به ستون","ستون کج","ستون آسیب","ستون ها ضربه"]; return int(contains_any(t,p))
def feat_golgir_rang(r): t=r["text_all"]; p=["گلگیر رنگ","رنگ گلگیر"]; return int(contains_any(t,p) and "گلگیر بدون رنگ" not in t and "گلگیر بی رنگ" not in t)
def feat_golgir_bedun_rang(r): t=r["text_all"]; p=["گلگیر بدون رنگ","گلگیر بی رنگ"]; return int(contains_any(t,p))
def feat_golgir_taviz(r): t=r["text_all"]; p=["گلگیر تعویض","تعویض گلگیر","گلگیر عوض شده"]; return int(contains_any(t,p))
def feat_fanni_salem(r): t=r["text_all"]; p=["فنی سالم","از نظر فنی سالم","از لحاظ فنی سالم"]; return int(contains_any(t,p))
def feat_fanni_niyaz_tamir(r): t=r["text_all"]; p=["فنی نیاز به تعمیر","نیاز به تعمیر فنی","مشکل فنی","ایراد فنی","خرج فنی"]; return int(contains_any(t,p))
def feat_fanni_tamir_shode(r): t=r["text_all"]; p=["فنی تعمیر شده","فنی تازه تعمیر","تعمیر اساسی فنی","تعمیر کلی فنی"]; return int(contains_any(t,p) or ("فنی" in t and "تعمیر" in t))
def feat_gearbox_salem(r): t=r["text_all"]; p=["گیربکس سالم","گیر بکس سالم","گیربکس بدون مشکل","گیربکس بی صدا"]; return int(contains_any(t,p))
def feat_gearbox_niyaz_tamir(r): t=r["text_all"]; p=["گیربکس نیاز به تعمیر","گیر بکس نیاز به تعمیر","گیربکس خرج","گیر بکس خرج","گیربکس خراب","مشکل گیربکس","ایراد گیربکس"]; return int(contains_any(t,p))
def feat_yek_lakke_rang(r): t=normalize_persian_text(r.get("body_status",""))+" "+r["text_all"]; p=["یک لکه رنگ","1 لکه رنگ","۱ لکه رنگ"]; return int(contains_any(t,p))
def feat_do_lakke_rang(r): t=normalize_persian_text(r.get("body_status",""))+" "+r["text_all"]; p=["دو لکه رنگ","2 لکه رنگ","۲ لکه رنگ"]; return int(contains_any(t,p))
def feat_dor_rang(r): t=normalize_persian_text(r.get("body_status",""))+" "+r["text_all"]; p=["دور رنگ","دوررنگ","تمام رنگ","تمام‌رنگ","تمام رنگی"]; return int(contains_any(t,p))

def get_feature_functions() -> Dict[str, Callable[[pd.Series], int]]:
    return {
        "درب رنگ": feat_darb_rang,
        "درب تعویض": feat_darb_taviz,
        "درب بدون رنگ": feat_darb_bedun_rang,
        "کاپوت تعویض": feat_kaput_taviz,
        "کاپوت بدون رنگ": feat_kaput_bedun_rang,
        "درب صندوق رنگ": feat_darb_sandogh_rang,
        "درب صندوق بدون رنگ": feat_darb_sandogh_bedun_rang,
        "شاسی ضرب خوردگی": feat_shasi_zarbe,
        "شاسی پلمپ": feat_shasi_pelomp,
        "شاسی سالم": feat_shasi_salem,
        "سقف رنگ": feat_saghf_rang,
        "سقف بدون رنگ": feat_saghf_bedun_rang,
        "ستون ها سالم": feat_sotun_salem,
        "ستون ها ضرب دیدگی": feat_sotun_zarbe,
        "گلگیر رنگ شدگی": feat_golgir_rang,
        "گلگیر بدون رنگ": feat_golgir_bedun_rang,
        "گلگیر تعویض": feat_golgir_taviz,
        "فنی سالم": feat_fanni_salem,
        "فنی نیاز به تعمیر": feat_fanni_niyaz_tamir,
        "فنی تعمیر شده": feat_fanni_tamir_shode,
        "گیربکس سالم": feat_gearbox_salem,
        "گیر بکس نیاز به تعمیر": feat_gearbox_niyaz_tamir,
        "یک لکه رنگ": feat_yek_lakke_rang,
        "دو لکه رنگ": feat_do_lakke_rang,
        "دور رنگ": feat_dor_rang,
    }

def apply_feature_functions(df: pd.DataFrame, feature_funcs: Dict[str, Callable[[pd.Series], int]]) -> pd.DataFrame:
    if df.empty:
        print("[WARN] Empty dataframe passed to apply_feature_functions")
        return df
    df = df.copy()
    for col, func in feature_funcs.items():
        try:
            df[col] = df.apply(func, axis=1).astype(int)
        except Exception as e:
            print(f"[ERROR] Failed feature '{col}': {e}")
            df[col] = 0
    return df


In [35]:
uploaded = files.upload()

frames = []
for fname in uploaded.keys():
    df_raw = load_excel_or_csv(fname)
    if df_raw.empty:
        continue
    df_uni = unify_generic(df_raw, source_name=os.path.splitext(os.path.basename(fname))[0])
    if not df_uni.empty:
        frames.append(df_uni)

if not frames:
    raise RuntimeError("not found")

master = pd.concat(frames, ignore_index=True)
print(f"[INFO] Master shape before filters: {master.shape}")

Saving car_bama_final.csv to car_bama_final (3).csv
Saving car_dataset_final.csv to car_dataset_final (3).csv
[WARN] Not Excel or failed: car_bama_final (3).csv -> File is not a zip file
[INFO] Loaded CSV: car_bama_final (3).csv enc=utf-8-sig shape=(846, 13)
[WARN] Not Excel or failed: car_dataset_final (3).csv -> File is not a zip file
[INFO] Loaded CSV: car_dataset_final (3).csv enc=utf-8-sig shape=(11314, 12)
[INFO] Master shape before filters: (12160, 10)


In [36]:
master = filter_finance_and_dealer(master)

[INFO] Removing 27 finance/dealer-like ads


In [37]:
before_neg = len(master)
master = filter_negotiable(master)
after_neg = len(master)
print(f"[CHECK] Negotiable removed: {before_neg - after_neg}")

[INFO] Removing 59 negotiable-price ads
[CHECK] Negotiable removed: 59


In [38]:
master = add_text_all(master)
feat_funcs = get_feature_functions()
final_df = apply_feature_functions(master, feat_funcs)

In [39]:
probe_cols = [c for c in ["price_raw","price","title","description"] if c in final_df.columns]
if probe_cols:
    probe_text = (final_df[probe_cols].astype(str).agg(" ".join, axis=1)).apply(normalize_persian_text)
    remain = probe_text.str.contains("توافق", na=False).sum()
    print(f"[CHECK] Rows still containing 'توافق*': {int(remain)}")

print(f"[INFO] Final shape after features: {final_df.shape}")


[CHECK] Rows still containing 'توافق*': 0
[INFO] Final shape after features: (12074, 36)


In [40]:
csv_path = "cars_features_merged_no_negotiable.csv"
xlsx_path = "cars_features_merged_no_negotiable.xlsx"

try:
    final_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"[INFO] Saved CSV: {csv_path}")
except Exception as e:
    print(f"[ERROR] Could not save CSV: {e}")

try:
    final_df.to_excel(xlsx_path, index=False)
    print(f"[INFO] Saved Excel: {xlsx_path}")
except Exception as e:
    print(f"[ERROR] Could not save Excel: {e}")

# دانلود
try:
    files.download(csv_path)
    files.download(xlsx_path)
except Exception as e:
    print(f"[WARN] Auto-download failed (use Files sidebar): {e}")


[INFO] Saved CSV: cars_features_merged_no_negotiable.csv
[INFO] Saved Excel: cars_features_merged_no_negotiable.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>